# tlc-analytics walkthrough

A short tour of the **pure-pandas reference core** and the **real pandas-vs-DuckDB bake-off** on seeded synthetic NYC-taxi-like data.

Everything here runs with only numpy / pandas + stdlib; the DuckDB comparison is *measured* when `duckdb` is installed and skips cleanly otherwise. No download, no engine cluster.

In [ ]:
import tlc
from tlc.demo import run_demo, synthesize_trips
from tlc.clean import clean_trips
from tlc import marts
tlc.__version__

## 1. Run the reproducible demo

`run_demo(0)` synthesizes trips, drives the real cleaner + marts, times the build with the real benchmark harness, runs the engine bake-off, and returns the headline numbers.

In [ ]:
result = run_demo(0)
result

## 2. Build the cleaned frame and the new marts

The three added marts: trip-duration buckets, revenue per day, and IQR anomaly flags.

In [ ]:
trips = clean_trips(synthesize_trips(0))
len(trips)

In [ ]:
marts.trip_duration_buckets(trips)

In [ ]:
marts.revenue_by_day(trips).head()

In [ ]:
flags = marts.anomaly_flags(trips)
int(flags['is_outlier'].sum()), len(flags)

## 3. Partition paths (pure stdlib, no I/O)

`iter_partitions` yields the expected `year=/month=` lake paths.

In [ ]:
list(tlc.iter_partitions('data/raw/yellow', [2023], [1, 2, 3]))

## 4. The real engine bake-off

`bake_off` times the pandas mart and, when DuckDB is importable, the equivalent SQL in-process. The result is a measured ranking; if DuckDB is absent the duckdb row reads `engine unavailable`.

In [ ]:
print('duckdb available:', tlc.duckdb_available())
tlc.bake_off(trips, query='hourly_demand')